# LightFM Model

### Unit 1 - 환경 및 공통 설정

In [ ]:
# Unit 1
from pathlib import Path
import random

import numpy as np
import pandas as pd

from lightfm import LightFM
from lightfm.data import Dataset
from lightfm.cross_validation import random_train_test_split
from lightfm.evaluation import precision_at_k, recall_at_k

SEED = 42
TEST_RATIO = 0.2
EPOCHS = 30
NUM_THREADS = 2

random.seed(SEED)
np.random.seed(SEED)

# DATA_PATH = Path("review_by_llm.csv")
print({"seed": SEED, "test_ratio": TEST_RATIO, "epochs": EPOCHS})

{'seed': 42, 'test_ratio': 0.2, 'epochs': 30}


### Unit 2 - 데이터 로드 및 필수 컬럼 검증

In [12]:
# 데이터 프레임 로드
review_df = pd.read_csv("review_by_llm.csv")
recipe_df = pd.read_csv("recipe_fix.csv")
ingredient_alias_df = pd.read_csv("recipe_ingredient_alias.csv")

recipe_cols = [
    "RCP_SNO", "CKG_NM", "INQ_CNT", "SRAP_CNT", "CKG_MTH_ACTO_NM",
    "CKG_STA_ACTO_NM", "CKG_MTRL_ACTO_NM", "CKG_KND_ACTO_NM",
    "CKG_INBUN_NM", "CKG_DODF_NM", "CKG_TIME_NM",
]
ingredient_alias_cols = [
    "RCP_SNO", "CKG_NM", "ingredients_raw", "aliases_matched",
    "ingredients_normalized", "others_count", "others_items",
    "basic_count", "basic_items",
]
review_cols = [
    "recipe_id", "group_id", "star_count", "content",
    "positive", "negative", "star_norm",
]

for name, frame, cols in [
    ("recipe_fix.csv", recipe_df, recipe_cols),
    ("recipe_ingredient_alias.csv", ingredient_alias_df, ingredient_alias_cols),
    ("review_by_llm.csv", review_df, review_cols),
]:
    missing_cols = [c for c in cols if c not in frame.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns in {name}: {missing_cols}")

recipe_df = recipe_df[recipe_cols].copy()
ingredient_alias_df = ingredient_alias_df[ingredient_alias_cols].copy()
review_df = review_df[review_cols].copy()

print("recipe_cols:", recipe_df.columns.tolist())
print("ingredient_alias_cols:", ingredient_alias_df.columns.tolist())
print("review_cols:", review_df.columns.tolist())

# 이후 유닛 호환용 별칭 (유저/레시피 축 재구성은 이후 셀에서 수행)
df = review_df.copy()

recipe_cols: ['RCP_SNO', 'CKG_NM', 'INQ_CNT', 'SRAP_CNT', 'CKG_MTH_ACTO_NM', 'CKG_STA_ACTO_NM', 'CKG_MTRL_ACTO_NM', 'CKG_KND_ACTO_NM', 'CKG_INBUN_NM', 'CKG_DODF_NM', 'CKG_TIME_NM']
ingredient_alias_cols: ['RCP_SNO', 'CKG_NM', 'ingredients_raw', 'aliases_matched', 'ingredients_normalized', 'others_count', 'others_items', 'basic_count', 'basic_items']
review_cols: ['recipe_id', 'group_id', 'star_count', 'content', 'positive', 'negative', 'star_norm']


### Unit 3 - interaction 타겟 생성
- 사용자 데이터에 필요한 값들 가공 처리
- 레시피 데이터에 필요한 값들 가공 처리

> 유저 데이터 처리

In [13]:
# 유저 데이터 전처리 - 별점 데이터 정리
# 강의 내용과 다르게 지금 수집한 내용 기준으로 정리, 필요한 경우 단계 나눠서 실험하면서 진행
# 값은 변환값 쓰지 않고 직접 계산해서 사용 
# star_count : 1 ~ 5점 데이터를 -1 ~ +1 사이 값으로 변환 ( -3 이후 * 1/2)
# review_df에서 별점 관련 데이터 드랍 및 필요한 칼럼만 남김

# 별점 데이터 관련 칼럼
star_related_cols = ["star_count", "star_norm"]

# 별점 값 가공해서 신규 컬럼으로 추가 (나중에 해당 부분에서 계산할 수 있도록)
# star_count 값을 -1 ~ +1 구간으로 변환해서 star 컬럼에 추가
# 변환식: ((star_count - 3) / 2)
review_df["star"] = review_df["star_count"].astype(float).apply(lambda x: (x - 3) / 2)


# review_df에서 별점 관련 칼럼 드랍
review_df = review_df.drop(columns=star_related_cols, errors="ignore")

# 필요한 값만 가공해서 사용: 여기서는 별점 관련 데이터가 사라지고
# 남은 칼럼만 유지됨을 확인
print("가공/드랍 이후 review_df 칼럼:", review_df.columns.tolist())

가공/드랍 이후 review_df 칼럼: ['recipe_id', 'group_id', 'content', 'positive', 'negative', 'star']


In [14]:
# 유저 데이터 전처리 - 감성 데이터 정리 
# 긍정 - 부정 값을 감성분석 값으로 사용 (긍정인 경우 +, 부정인 경우 - 나오도록 처리)
# 컬럼은 sentiment 컬럼으로 추가하고 기존 감성분석 컬럼은 제거 

# 감성 데이터 전처리
# 'positive', 'negative' 값을 활용하여 sentiment 컬럼 생성
# sentiment = positive - negative (긍정은 +, 부정은 -)
review_df["sentiment"] = review_df["positive"].astype(float) - review_df["negative"].astype(float)

# 기존 감성 관련 컬럼(drop)
sentiment_related_cols = ["positive", "negative", "neutral", "compound"]
review_df = review_df.drop(columns=sentiment_related_cols, errors="ignore")

# 결과 칼럼 확인
print("감성 처리/드랍 이후 review_df 칼럼:", review_df.columns.tolist())

감성 처리/드랍 이후 review_df 칼럼: ['recipe_id', 'group_id', 'content', 'star', 'sentiment']


In [15]:
# 리뷰 내용은 일단 제거 (content 컬럼)
# 리뷰 내용 제거 (content 컬럼 드랍)
review_df = review_df.drop(columns=["content"], errors="ignore")
print("content 컬럼 제거 후 review_df 칼럼:", review_df.columns.tolist())

content 컬럼 제거 후 review_df 칼럼: ['recipe_id', 'group_id', 'star', 'sentiment']


> 레시피 데이터 처리

In [17]:
# recipe_df, ingredient_alias_df를 RCP_SNO 기준으로 병합
# 중복 제거의 의미는 "중복 행"이 아니라 "겹치는 컬럼명" 처리(CKG_NM 등)

# recipe_df에 이미 있는 공통 컬럼은 ingredient_alias_df에서 제외하고 머지
recipe_base_cols = set(recipe_df.columns)
alias_merge_cols = [
    col for col in ingredient_alias_df.columns
    if col == "RCP_SNO" or col not in recipe_base_cols
]

ingredient_alias_for_merge = ingredient_alias_df[alias_merge_cols].copy()

recipe_df = recipe_df.merge(ingredient_alias_for_merge, on="RCP_SNO", how="left")

print("재료머지 이후 recipe_df 칼럼:", recipe_df.columns.tolist())
print(recipe_df.head())

재료머지 이후 recipe_df 칼럼: ['RCP_SNO', 'CKG_NM', 'INQ_CNT', 'SRAP_CNT', 'CKG_MTH_ACTO_NM', 'CKG_STA_ACTO_NM', 'CKG_MTRL_ACTO_NM', 'CKG_KND_ACTO_NM', 'CKG_INBUN_NM', 'CKG_DODF_NM', 'CKG_TIME_NM', 'ingredients_raw', 'aliases_matched', 'ingredients_normalized', 'others_count', 'others_items', 'basic_count', 'basic_items']
   RCP_SNO   CKG_NM  INQ_CNT  SRAP_CNT CKG_MTH_ACTO_NM CKG_STA_ACTO_NM  \
0  7016814     된장수육     1396         1              삶기             술안주   
1  7016815  우거지 감자탕     4008        29             끓이기              해장   
2  7016816     만두전골     6350         6             끓이기            손님접대   
3  7016817    무수분보쌈     1829         6              삶기            손님접대   
4  7016820   참치 카나페     4259        10              기타              일상   

  CKG_MTRL_ACTO_NM CKG_KND_ACTO_NM CKG_INBUN_NM CKG_DODF_NM CKG_TIME_NM  \
0             돼지고기            메인반찬          2인분          고급       2시간이내   
1             돼지고기             국/탕          4인분          중급       2시간이내   
2            가

In [19]:
recipe_df.columns

Index(['RCP_SNO', 'CKG_NM', 'INQ_CNT', 'SRAP_CNT', 'CKG_MTH_ACTO_NM',
       'CKG_STA_ACTO_NM', 'CKG_MTRL_ACTO_NM', 'CKG_KND_ACTO_NM',
       'CKG_INBUN_NM', 'CKG_DODF_NM', 'CKG_TIME_NM', 'ingredients_raw',
       'aliases_matched', 'ingredients_normalized', 'others_count',
       'others_items', 'basic_count', 'basic_items'],
      dtype='object')

In [20]:
# 레시피 컬럼 개별 정리 
# 사용할 컬럼의 경우 컬럼 이름 수정 
# RCP_SNO -> recipe_id
# CKG_NM -> recipe_name
# INQ_CNT -> view_count
# SRAP_CNT -> scrap_count
# CKG_MTH_ACTO_NM -> cooking_method
# CKG_STA_ACTO_NM -> cooking_category
# CKG_MTRL_ACTO_NM -> main_ingred
# CKG_INBUN_NM -> dishes
# CKG_DODF_NM -> cooking_level
# CKG_TIME_NM -> cooking_time
# ingredients_raw -> 컬럼 드랍 (정규화 이름만 사용)
# aliases_matched -> aliases
# ingredients_normalized -> ingredients
# others_items -> 컬럼 드랍 (정규화된 재료 기준으로 유사도 비교)
# basic_items -> 컬럼 드랍 (기본 재료가 몇개 있는지만 체크)
# 위 기준으로 데이터 정리

# 컬럼명 매핑 정의
column_rename_map = {
    "RCP_SNO": "recipe_id",
    "CKG_NM": "recipe_name",
    "INQ_CNT": "view_count",
    "SRAP_CNT": "scrap_count",
    "CKG_MTH_ACTO_NM": "cooking_method",
    "CKG_STA_ACTO_NM": "cooking_category",
    "CKG_MTRL_ACTO_NM": "main_ingred",
    "CKG_INBUN_NM": "dishes",
    "CKG_DODF_NM": "cooking_level",
    "CKG_TIME_NM": "cooking_time",
    "aliases_matched": "aliases",
    "ingredients_normalized": "ingredients"
}

# 드랍할 컬럼 정의
columns_to_drop = [
    "ingredients_raw",       # 원본 재료명
    "others_items",          # 기타 재료 리스트
    "basic_items"            # 기본 재료 리스트
]

# 컬럼명 변경
recipe_df = recipe_df.rename(columns=column_rename_map)

# 필요없는 컬럼 드랍
recipe_df = recipe_df.drop(columns=[col for col in columns_to_drop if col in recipe_df.columns])

# 결과 컬럼 확인
print("정리 이후 recipe_df 컬럼:", recipe_df.columns.tolist())

정리 이후 recipe_df 컬럼: ['recipe_id', 'recipe_name', 'view_count', 'scrap_count', 'cooking_method', 'cooking_category', 'main_ingred', 'CKG_KND_ACTO_NM', 'dishes', 'cooking_level', 'cooking_time', 'aliases', 'ingredients', 'others_count', 'basic_count']


In [ ]:
# Unit 3
TARGET_MODE = "binary"  # binary | star_norm | sentiment | hybrid

if TARGET_MODE == "binary":
    df["interaction_value"] = 1.0
elif TARGET_MODE == "star_norm":
    df["interaction_value"] = df["star_norm"].astype(float)
elif TARGET_MODE == "sentiment":
    df["interaction_value"] = (df["positive"] - df["negative"]).astype(float)
elif TARGET_MODE == "hybrid":
    df["interaction_value"] = (df["star_norm"] + (df["positive"] - df["negative"]))
else:
    raise ValueError(f"Unsupported TARGET_MODE: {TARGET_MODE}")

print("target_mode:", TARGET_MODE)
print(df["interaction_value"].describe())

### Unit 4 - ID 매핑 및 Dataset 구성
- Why: user/item 내부 인덱스 매핑을 고정한다.
- Input: `group_id`, `recipe_id`
- Output: LightFM Dataset
- Check: user/item 개수 출력
- DoD: `dataset.fit()` 완료

In [ ]:
# Unit 4
user_ids = df["group_id"].astype(str).unique().tolist()
item_ids = df["recipe_id"].astype(str).unique().tolist()

dataset = Dataset()
dataset.fit(users=user_ids, items=item_ids)

print({"users": len(user_ids), "items": len(item_ids)})

### Unit 5 - interactions matrix 생성
- Why: 학습/평가 입력 sparse matrix를 만든다.
- Input: `(user, item, interaction_value)`
- Output: interactions matrix
- Check: matrix shape, nnz 출력
- DoD: interactions 생성 및 sparsity 계산 가능

In [ ]:
# Unit 5
triples = list(
    zip(
        df["group_id"].astype(str),
        df["recipe_id"].astype(str),
        df["interaction_value"].astype(float),
    )
)

interactions, _ = dataset.build_interactions(triples)
num_users, num_items = interactions.shape
density = interactions.nnz / (num_users * num_items) if num_users and num_items else 0.0

print({"shape": interactions.shape, "nnz": interactions.nnz, "density": density})

### Unit 6 - train/test 분할
- Why: 오프라인 평가를 위한 검증 셋을 분리한다.
- Input: interactions
- Output: train, test
- Check: train/test nnz 출력
- DoD: 재현 가능한 분할(random_state 고정) 완료

In [ ]:
# Unit 6
train, test = random_train_test_split(
    interactions,
    test_percentage=TEST_RATIO,
    random_state=np.random.RandomState(SEED),
)

print({"train_nnz": train.nnz, "test_nnz": test.nnz})

### Unit 7 - LightFM 학습
- Why: 랭킹 모델을 학습해 추천 점수를 생성한다.
- Input: train matrix
- Output: 학습된 model
- Check: 학습 완료, train Precision@5 추이 확인
- DoD: 지정 epoch 학습이 종료되고 최소 지표가 산출됨

In [ ]:
# Unit 7
model = LightFM(loss="warp", random_state=SEED)
train_p5_history = []

for _ in range(EPOCHS):
    model.fit_partial(train, epochs=1, num_threads=NUM_THREADS)
    p5 = precision_at_k(model, train, k=5).mean()
    train_p5_history.append(float(p5))

print({"last_train_precision@5": train_p5_history[-1] if train_p5_history else None})

### Unit 8 - Precision/Recall 평가
- Why: 모델의 추천 품질을 수치화한다.
- Input: model, test matrix
- Output: Precision@K, Recall@K
- Check: K=5/10 지표 출력
- DoD: 핵심 지표 4개(p@5, p@10, r@5, r@10) 기록

In [ ]:
# Unit 8
metrics = {
    "precision@5": float(precision_at_k(model, test, k=5).mean()),
    "precision@10": float(precision_at_k(model, test, k=10).mean()),
    "recall@5": float(recall_at_k(model, test, k=5).mean()),
    "recall@10": float(recall_at_k(model, test, k=10).mean()),
}
print(metrics)

### Unit 9 - baseline 비교
- Why: LightFM 성능 우위를 판단한다.
- Input: LightFM 지표, baseline 지표
- Output: 개선/동등/열위 판정
- Check: 비교표 및 판정 로그
- DoD: Go/No-Go 초안 기준으로 결과 상태가 결정됨

In [ ]:
# Unit 9
# TODO: baseline 계산 로직(인기 기반)을 연결해 아래 dict를 실제 값으로 채운다.
baseline_metrics = {
    "precision@10": None,
    "recall@10": None,
}

go_no_go = "pending"
if baseline_metrics["precision@10"] is not None and baseline_metrics["recall@10"] is not None:
    if (
        metrics["precision@10"] >= baseline_metrics["precision@10"]
        and metrics["recall@10"] >= baseline_metrics["recall@10"]
    ):
        go_no_go = "go"
    else:
        go_no_go = "no_go"

print({"baseline": baseline_metrics, "decision": go_no_go})

### Unit 10 - 실험 리포트 기록
- Why: 실험 설정/결과/판단을 문서 기준으로 누적한다.
- Input: 실행 설정, metrics, baseline 비교 결과
- Output: 구조화된 실험 요약
- Check: 필수 필드 누락 여부
- DoD: 결과가 재현 가능한 형식으로 남음

In [ ]:
# Unit 10
experiment_report = {
    "data_path": str(DATA_PATH),
    "target_mode": TARGET_MODE,
    "seed": SEED,
    "test_ratio": TEST_RATIO,
    "epochs": EPOCHS,
    "loss": "warp",
    "matrix": {
        "num_users": int(interactions.shape[0]),
        "num_items": int(interactions.shape[1]),
        "nnz": int(interactions.nnz),
    },
    "metrics": metrics,
    "decision": go_no_go,
}

print(experiment_report)